# Generación de SBOMs

## Objetivo

Este notebook explica el proceso para generar **Software Bill of Materials (SBOMs)** para un conjunto de repositorios de código y cómo analizar los resultados de manera centralizada. Utilizaremos la herramienta **Syft** para la generación y la librería **Pandas** en Python para el análisis.

El flujo de trabajo completo está diseñado para ser reproducible y se basa en las siguientes etapas:

1.  **Descubrimiento de Repositorios**: Identificar los proyectos a los que se les generará un SBOM.
2.  **Generación de SBOMs**: Ejecutar Syft para analizar cada repositorio y crear un SBOM en formato JSON.
3.  **Análisis de Resultados**: Cargar todos los SBOMs generados en un DataFrame de Pandas para su inspección y análisis.


## Estructura de Directorios

El proyecto sigue una estructura organizada para separar los datos de entrada, los scripts y los resultados:

```
ciberseguridad_2026/
├── data/
│   ├── repos/      # Directorio para los repositorios a analizar
│   └── results/    # Directorio para los SBOMs generados en JSON
├── nbs/            # Notebooks de Jupyter
└── scripts/        # Scripts de automatización
    └── generate_sboms.py
```

## Paso 1: Generación de SBOMs

El primer paso es ejecutar el script `generate_sboms.py`, que se encarga de orquestar la generación de los SBOMs.

Este script realiza las siguientes acciones:

1.  **Busca repositorios**: Escanea el directorio `data/repos` en busca de subdirectorios que contengan código fuente.
2.  **Ejecuta Syft**: Para cada repositorio encontrado, invoca a `syft` para analizar su contenido.
3.  **Guarda los resultados**: El SBOM generado para cada repositorio se guarda como un archivo JSON en el directorio `data/results`.

In [1]:
#| eval: false
!python ../../scripts/generate_sboms.py

INFO | Usando Syft CLI: /usr/local/bin/syft
INFO | [1/5] Procesando repositorio data/repos/cache
INFO | SBOM guardado en data/results/cache-sbom.json
INFO | [2/5] Procesando repositorio data/repos/checkout
INFO | SBOM guardado en data/results/checkout-sbom.json
INFO | [3/5] Procesando repositorio data/repos/github-script
INFO | SBOM guardado en data/results/github-script-sbom.json
INFO | [4/5] Procesando repositorio data/repos/starter-workflows
INFO | SBOM guardado en data/results/starter-workflows-sbom.json
INFO | [5/5] Procesando repositorio data/repos/toolkit
INFO | SBOM guardado en data/results/toolkit-sbom.json
INFO | Resumen final | total_repos=5 | repos_generados=5 | archivos_generados=5 | omitidos=0 | errores=0


## Paso 2: Análisis de los SBOMs con Pandas

Una vez que los SBOMs han sido generados, podemos cargarlos en un DataFrame de Pandas para analizarlos. Esto nos permite tener una visión consolidada de todas las dependencias y artefactos encontrados en los diferentes repositorios.

El siguiente código se encarga de:

1.  **Localizar los archivos JSON**: Busca todos los archivos `.json` en el directorio `data/results`.
2.  **Cargar y normalizar los datos**: Para cada archivo JSON, lo carga y utiliza `pd.json_normalize` para aplanar la estructura anidada de los artefactos.
3.  **Añadir información del repositorio**: Agrega una columna `repo` para identificar a qué repositorio pertenece cada artefacto.
4.  **Concatenar los resultados**: Une todos los DataFrames individuales en uno solo.

In [2]:
import pandas as pd
from pathlib import Path
import json

path = Path("../../data/results")

df_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))[
                          "artifacts"]).assign(repo=file.stem)
        for file in path.glob("*-sbom.json")
    ],
    ignore_index=True
)

df_all.head()

,id,name,version,type,foundBy,locations,licenses,language,cpes,purl,...,metadata.dependencies.binary,metadata.dependencies.mkdirp,metadata.dependencies.isexe,metadata.dependencies.ansi-styles,metadata.dependencies.compress-commons,metadata.dependencies.sprintf-js,metadata.dependencies.argparse,metadata.dependencies.esprima,metadata.dependencies.js-yaml,metadata.dependencies.jsonschema
0,5da7cb142b040b4f,./,UNKNOWN,github-action,github-actions-usage-cataloger,[{'path': '/.github/workflows/integration.yml'...,[],,"[{'cpe': 'cpe:2.3:a:.\/:.\/:*:*:*:*:*:*:*:*', ...",,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40efa51e9a36ba6b,./,UNKNOWN,github-action,github-actions-usage-cataloger,[{'path': '/.github/workflows/pull-request-tes...,[],,"[{'cpe': 'cpe:2.3:a:.\/:.\/:*:*:*:*:*:*:*:*', ...",,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0a39fc8197e46d1e,./.github/actions/install-dependencies,UNKNOWN,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/check-dist.yml',...",[],,[{'cpe': 'cpe:2.3:a:.\/.github\/actions\/insta...,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,f1104ad63338d673,./.github/actions/install-dependencies,UNKNOWN,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/ci.yml', 'access...",[],,[{'cpe': 'cpe:2.3:a:.\/.github\/actions\/insta...,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,d212bcd98d27c39b,./.github/actions/install-dependencies,UNKNOWN,github-action,github-actions-usage-cataloger,[{'path': '/.github/workflows/integration.yml'...,[],,[{'cpe': 'cpe:2.3:a:.\/.github\/actions\/insta...,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Exploración de los Resultados

### Cantidad de artefactos según su tipo

In [3]:
df_all['type'].value_counts()

type
npm                       455
github-action              89
github-action-workflow      1
binary                      1
Name: count, dtype: int64

#### Lenguaje principales de los artefactos

In [4]:
df_all['language'].value_counts()

language
javascript    455
               91
Name: count, dtype: int64

Los 331 no presentan lenguaje porque seguramente son de tipo Gibhub actions (tal como muestra la _query_ anterior)

### Se busca las relaciones entre dependencias más frecuentes

In [5]:

artifact_relationships_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))[
                          "artifactRelationships"]).assign(repo=file.stem)
        for file in path.glob("*-sbom.json")
    ],
    ignore_index=True
)

artifact_relationships_all.value_counts("type")

type
dependency-of    825
contains         546
evident-by       546
Name: count, dtype: int64

### Artefactos por cada repositorio

In [6]:
df_all['repo'].value_counts()

repo
toolkit-sbom              355
github-script-sbom         57
cache-sbom                 56
checkout-sbom              50
starter-workflows-sbom     28
Name: count, dtype: int64

Se muestra que el proyecto principal (_FastAPI_) es el que presenta mayor cantidad de artefactos.

### Licencias encontradas en los artefactos

In [7]:
df_all['licenses'].explode().value_counts().head(10)

licenses
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/packages/artifact/package-lock.json', 'accessPath': '/packages/artifact/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                  103
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/packages/attest/package-lock.json', 'accessPath': '/packages/attest/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                       66
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/package-lock.json', 'accessPath': '/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                                                       63
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/packages/cache/package-lock.json', 'accessPath': '/packages/cache/package-lock.json', 'annotations': {'evidence': 'primary'}}]}   

Se puede apreciar que no se encontraron licencias en ninguno de los artefactos.

### Paquetes compartidos entre repos

In [8]:
common = (
    df_all.dropna(subset=["purl"])
    .groupby("purl")["repo"].nunique()
    .sort_values(ascending=False)
)
common.head(20)

purl
pkg:npm/tunnel@0.0.6                          5
                                              4
pkg:github/actions/setup-node@v4              4
pkg:github/github/codeql-action@v3#analyze    3
pkg:github/github/codeql-action@v3#init       3
pkg:npm/uuid@8.3.2                            3
pkg:npm/%40actions/http-client@3.0.2          3
pkg:npm/undici@6.24.1                         3
pkg:npm/concat-map@0.0.1                      3
pkg:npm/strnum@2.1.2                          2
pkg:npm/minimatch@3.1.2                       2
pkg:npm/https-proxy-agent@7.0.6               2
pkg:npm/fast-content-type-parse@3.0.0         2
pkg:npm/http-proxy-agent@7.0.2                2
pkg:npm/events@3.3.0                          2
pkg:npm/debug@4.4.3                           2
pkg:npm/brace-expansion@1.1.12                2
pkg:npm/bottleneck@2.19.5                     2
pkg:npm/before-after-hook@4.0.0               2
pkg:npm/%40actions/exec@1.1.1                 2
Name: repo, dtype: int64

## Cómo Añadir Nuevos Repositorios al Análisis

El proyecto está diseñado para que sea sencillo añadir nuevos repositorios al proceso de generación de SBOMs. El sistema utiliza **Git submodules** para gestionar los repositorios externos.

#### 1. Edita el archivo `data/repos.json`

Abre `data/repos.json` y agrega nuevos repositorios con esta estructura:

```json
{
  "url": "URL_DEL_REPOSITORIO_EN_GITHUB.git",
  "path": "data/repos/NOMBRE_DEL_REPOSITORIO"
}
```

- `url`: La URL `.git` del repositorio
- `path`: Ruta local donde se clonará (usa `data/repos/{nombre}`)

#### 2. Ejecuta en la terminal (esto clona los repositorios y los agrega como submódulos):

```bash
uv run scripts/add_submodules.py && git submodule update --init --recursive
```

#### 3. Ejecuta la generación de SBOMs nuevamente:

```bash
uv run scripts/generate_sboms.py
```

O desde el notebook:
- Reinicia el kernel
- Ejecuta la celda con `!python ../../scripts/generate_sboms.py`
